# Manhwa Studio — Google Colab + Google Drive

Versão independente para renderizar com FFmpeg no ambiente do Colab e ler/gravar arquivos diretamente no Google Drive. A interface web original continua separada.

**Uso autorizado:** processe somente vídeos que você possui ou tem autorização para baixar e editar.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
WORK = Path('/content/drive/MyDrive/ManhwaStudio')
INPUT = WORK / 'entrada'
OUTPUT = WORK / 'exportados'
INPUT.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Entrada:', INPUT)
print('Saída:', OUTPUT)

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!ffmpeg -version | head -n 2
!nvidia-smi || true

In [ ]:
!git clone --depth 1 --branch google-colab-drive https://github.com/manhwavaz1-lab/manhwa-recreator-studio.git /content/manhwa-recreator-studio
%cd /content/manhwa-recreator-studio
!python -m pip install -q yt-dlp

## 1. Vincular vídeo e moldura
Na interface interativa abaixo, você terá campos separados para selecionar arquivos do Google Drive ou enviar um vídeo e uma moldura. Depois, clique em **Renderizar vídeo** e use **Baixar MP4** para baixar o resultado.


In [ ]:
from pathlib import Path
videos = sorted([*INPUT.glob('*.mp4'), *INPUT.glob('*.mov'), *INPUT.glob('*.mkv'), *INPUT.glob('*.webm')])
frames = sorted([*INPUT.glob('*.png'), *INPUT.glob('*.jpg'), *INPUT.glob('*.jpeg')])
print('Vídeos:', [p.name for p in videos])
print('Molduras:', [p.name for p in frames])

# Edite estes caminhos se houver mais de um arquivo:
VIDEO = str(videos[0]) if videos else ''
FRAME = str(frames[0]) if frames else ''
print('VIDEO =', VIDEO, '\\nFRAME =', FRAME)

## 2. Opcional: baixar vídeo do YouTube
Use apenas quando tiver direito/autorização para baixar o conteúdo. Deixe `YOUTUBE_URL` vazio se já colocou o MP4 no Drive.

In [ ]:
YOUTUBE_URL = ''  # cole aqui o link, se necessário
if YOUTUBE_URL.strip():
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'yt_dlp', '-f', 'bv*+ba/b', '--merge-output-format', 'mp4', '-o', str(INPUT / 'youtube_source.%(ext)s'), YOUTUBE_URL], check=True)
    downloaded = sorted(INPUT.glob('youtube_source.*'), key=lambda p:p.stat().st_mtime, reverse=True)
    VIDEO = str(downloaded[0])
print('Vídeo selecionado:', VIDEO or 'Nenhum')

## 3. Interface de renderização
Selecione o vídeo, selecione a moldura, escolha o formato e clique em renderizar. O arquivo também será salvo no Google Drive.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files
import shutil, subprocess, sys

def scan_drive_files():
    vids = sorted([*INPUT.glob('*.mp4'), *INPUT.glob('*.mov'), *INPUT.glob('*.mkv'), *INPUT.glob('*.webm')])
    imgs = sorted([*INPUT.glob('*.png'), *INPUT.glob('*.jpg'), *INPUT.glob('*.jpeg')])
    return vids, imgs

video_select = widgets.Dropdown(description='Vincular vídeo:', options=[], layout=widgets.Layout(width='95%'))
frame_select = widgets.Dropdown(description='Vincular moldura:', options=[], layout=widgets.Layout(width='95%'))
video_upload = widgets.FileUpload(accept='.mp4,.mov,.mkv,.webm', multiple=False, description='Enviar vídeo')
frame_upload = widgets.FileUpload(accept='.png,.jpg,.jpeg', multiple=False, description='Enviar moldura')
refresh_btn = widgets.Button(description='Atualizar arquivos do Drive', icon='refresh')
render_btn = widgets.Button(description='Renderizar vídeo', button_style='success', icon='film')
download_btn = widgets.Button(description='Baixar MP4', button_style='info', icon='download', disabled=True)
format_select = widgets.Dropdown(options=['16:9','9:16'], value='16:9', description='Formato:')
encoder_select = widgets.Dropdown(options=['auto','nvenc','qsv','amf','cpu'], value='auto', description='Encoder:')
zoom_slider = widgets.IntSlider(value=100, min=50, max=200, step=5, description='Zoom:')
crop_check = widgets.Checkbox(value=True, description='Recortar para preencher')
status = widgets.Output()
OUT = str(OUTPUT / 'manhwa-studio-export.mp4')

def uploaded_bytes(widget):
    value = widget.value
    if not value: return None
    if isinstance(value, dict):
        item = next(iter(value.values()))
        return item['content']
    item = value[0]
    return item['content']

def save_upload(widget, allowed_name):
    data = uploaded_bytes(widget)
    if data is None: return
    name = allowed_name
    if isinstance(widget.value, dict):
        name = next(iter(widget.value.keys()))
    else:
        name = widget.value[0]['name']
    target = INPUT / Path(name).name
    target.write_bytes(bytes(data))
    refresh_options()
    print('Arquivo recebido:', target.name)

def refresh_options(*_):
    vids, imgs = scan_drive_files()
    video_select.options = [(p.name, str(p)) for p in vids]
    frame_select.options = [(p.name, str(p)) for p in imgs]
    if vids: video_select.value = str(vids[0])
    if imgs: frame_select.value = str(imgs[0])

def render_click(_):
    with status:
        clear_output()
        if not video_select.value:
            print('Selecione ou envie um vídeo primeiro.'); return
        if not frame_select.value:
            print('Selecione ou envie uma moldura primeiro.'); return
        cmd=[sys.executable, 'colab/renderer.py', '--video', video_select.value,
             '--frame', frame_select.value, '--output', OUT, '--format', format_select.value,
             '--encoder', encoder_select.value, '--fps', '30', '--zoom', str(zoom_slider.value)]
        if crop_check.value: cmd.append('--crop')
        try:
            subprocess.run(cmd, check=True)
            download_btn.disabled = False
            print('Renderização concluída. Arquivo salvo no Google Drive:', OUT)
        except subprocess.CalledProcessError as e:
            print('Falha na renderização:', e)

def download_click(_):
    if Path(OUT).exists():
        files.download(OUT)
    else:
        with status: print('Renderize o vídeo antes de baixar.')

video_upload.observe(lambda change: save_upload(video_upload, 'video'), names='value')
frame_upload.observe(lambda change: save_upload(frame_upload, 'frame'), names='value')
refresh_btn.on_click(refresh_options)
render_btn.on_click(render_click)
download_btn.on_click(download_click)
refresh_options()

display(widgets.HTML('<h3>Arquivos de entrada</h3><p>Selecione arquivos já existentes no Google Drive ou envie novos arquivos para a pasta ManhwaStudio/entrada.</p>'))
display(video_select, video_upload, frame_select, frame_upload, refresh_btn)
display(widgets.HTML('<h3>Configurações de renderização</h3>'))
display(format_select, encoder_select, zoom_slider, crop_check, render_btn, download_btn, status)
